# BTC Direction Classifier — EDA & Feature Engineering

This notebook walks through:
1. Price chart and return distribution
2. Feature engineering via `src/features.py`
3. Correlation heatmap of features
4. Target label balance check

> **Note:** All production logic lives in `src/`. This notebook is for
> exploration and portfolio presentation only.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from data_loader import load_btc
from features import build_features, chronological_split

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='darkgrid', palette='muted')

## 1  Load Raw Data

In [ ]:
df = load_btc()
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')
df.tail(3)

## 2  Price Chart

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(df.index, df['Close'], linewidth=1)
axes[0].set_ylabel('Close Price (USD)')
axes[0].set_title('BTC-USD Daily Close')

axes[1].bar(df.index, df['Volume'], width=1, alpha=0.6)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

fig.tight_layout()
plt.show()

## 3  Return Distribution

In [ ]:
ret = df['Close'].pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ret, bins=80, edgecolor='none')
axes[0].set_title('Daily Return Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)

axes[1].plot(ret.index, ret.rolling(30).std() * np.sqrt(365), linewidth=1)
axes[1].set_title('Annualised 30-Day Realised Volatility')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volatility')

fig.tight_layout()
plt.show()

print(f'Mean daily return : {ret.mean()*100:.3f}%')
print(f'Std  daily return : {ret.std()*100:.3f}%')
print(f'Skewness          : {ret.skew():.3f}')
print(f'Kurtosis          : {ret.kurt():.3f}')

## 4  Feature Engineering

In [ ]:
X, y = build_features(df)
print(f'Feature matrix  : {X.shape}')
print(f'Target balance  : {y.mean()*100:.1f}% up-days')
X.head(3)

### Target label check

A ~50/50 split is expected for a noisy series.  Any strong imbalance would
warrant class weighting in the model.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
y.value_counts().plot(kind='bar', ax=ax, rot=0)
ax.set_xticklabels(['Down (0)', 'Up (1)'])
ax.set_title('Target Label Distribution')
ax.set_ylabel('Count')
fig.tight_layout()
plt.show()

## 5  Feature Correlation Heatmap

In [ ]:
corr = X.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=False,
    linewidths=0.4,
    ax=ax,
)
ax.set_title('Feature Correlation Matrix')
fig.tight_layout()
plt.show()

## 6  Chronological Train/Test Split

We use an 80/20 chronological split.  The boundary date is printed below.

In [ ]:
X_train, X_test, y_train, y_test = chronological_split(X, y)

fig, ax = plt.subplots(figsize=(12, 3))
close_aligned = df.loc[X.index, 'Close']
split_date = X_test.index[0]

ax.plot(close_aligned.loc[X_train.index], label='Train', linewidth=1)
ax.plot(close_aligned.loc[X_test.index], label='Test', linewidth=1)
ax.axvline(split_date, color='red', linestyle='--', linewidth=1.5, label=f'Split: {split_date.date()}')
ax.set_title('Train / Test Split')
ax.set_ylabel('BTC Close (USD)')
ax.legend()
fig.tight_layout()
plt.show()

## 7  Feature Statistics (train set only)

Computed on training data only to avoid information leakage into any
normalisation step.

In [ ]:
X_train.describe().T.round(4)